
Retrieve the API key from Databricks secrets dbutils.secrets.get() securely fetches a secret stored in the Databricks Secret Scope.

In [0]:
api_key = dbutils.secrets.get(scope="stocks", key="api-key")

In [0]:
import time, requests, json
import pandas as pd
from datetime import datetime

api_key = dbutils.secrets.get(scope="stocks", key="api-key")

LANDING_DIR = "/Volumes/workspace/bronze/landing_zone/stocks/"

def fetch_and_land(ticker):
    # Fetch OHLCV and write each day as a JSON file to the landing zone
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": ticker,
        "outputsize": "compact",
        "apikey": api_key,
    }
    data = requests.get(url, params=params).json()

    if "Time Series (Daily)" not in data:
        print(f"[{ticker}] unexpected response: {data}")
        return

    ts = data["Time Series (Daily)"]
    for date, values in ts.items():
        record = {
        "ticker": ticker,
        "date": date,
        "open":         values["1. open"],
        "high":         values["2. high"],
        "low":          values["3. low"],
        "close":        values["4. close"],
        "volume":       values["5. volume"],
        "ingested_at":  datetime.utcnow().isoformat(),
        }
        ts_now = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        path = f"{LANDING_DIR}{ticker}_{date}_{ts_now}.json"
        with open(path, "w") as f:
            json.dump(record, f)

    print(f"[{ticker}] landed {len(ts)} files")

tickers = ["AAPL", "MSFT", "IBM"]
for t in tickers:
    fetch_and_land(t)
    time.sleep(15)